# Glue + Iceberg Demo (Local Playground)

This notebook runs inside the **AWS Glue 4.0 Docker container** (JupyterLab on port 8888).

It demonstrates end-to-end:
1. Reading raw CSV data from **MinIO** (S3-compatible)
2. Writing **Apache Iceberg** tables via the **Project Nessie** catalog
3. Querying and time-travelling Iceberg tables
4. Glue DynamicFrame API usage

### Prerequisites
- Playground stack running: `docker compose --env-file .env.playground up -d`
- JupyterLab open: http://localhost:8888?token=datahive
- MinIO init complete (buckets + sample data created)


In [ ]:
# Cell 1 — Verify environment
import os
import sys

print("Python:", sys.version)
print("MINIO_ENDPOINT:",  os.getenv('MINIO_ENDPOINT',  'http://minio:9000'))
print("NESSIE_URI:",      os.getenv('NESSIE_URI',      'http://nessie:19120/api/v2'))
print("ICEBERG_WAREHOUSE:",os.getenv('ICEBERG_WAREHOUSE','s3a://iceberg-warehouse/'))

In [ ]:
# Cell 2 — Build SparkSession with Iceberg + Nessie + MinIO
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT',    'http://minio:9000')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY',  'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY',  'minioadmin')
NESSIE_URI       = os.getenv('NESSIE_URI',        'http://nessie:19120/api/v2')
WAREHOUSE        = os.getenv('ICEBERG_WAREHOUSE', 's3a://iceberg-warehouse/')

spark = (
    SparkSession.builder
    .appName('glue-iceberg-notebook')
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.nessie',
            'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.nessie.catalog-impl',
            'org.apache.iceberg.nessie.NessieCatalog')
    .config('spark.sql.catalog.nessie.uri',          NESSIE_URI)
    .config('spark.sql.catalog.nessie.ref',          'main')
    .config('spark.sql.catalog.nessie.warehouse',    WAREHOUSE)
    .config('spark.sql.catalog.nessie.io-impl',
            'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.nessie.s3.endpoint',           MINIO_ENDPOINT)
    .config('spark.sql.catalog.nessie.s3.path-style-access',  'true')
    .config('spark.sql.catalog.nessie.s3.access-key-id',      MINIO_ACCESS_KEY)
    .config('spark.sql.catalog.nessie.s3.secret-access-key',  MINIO_SECRET_KEY)
    .config('spark.sql.defaultCatalog', 'nessie')
    .config('spark.hadoop.fs.s3a.impl',
            'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.endpoint',            MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',          MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',          MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access',   'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.checksum.validation', 'false')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('SparkSession ready. Version:', spark.version)

In [ ]:
# Cell 3 — Create Iceberg namespace
spark.sql('CREATE NAMESPACE IF NOT EXISTS nessie.datahive')
spark.sql('SHOW NAMESPACES IN nessie').show()

In [ ]:
# Cell 4 — Read products CSV from MinIO raw-landing/
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from datetime import datetime, timezone

PRODUCTS_SCHEMA = StructType([
    StructField('product_id',     StringType(),  False),
    StructField('product_name',   StringType(),  True),
    StructField('category',       StringType(),  True),
    StructField('price',          DoubleType(),  True),
    StructField('stock_quantity', IntegerType(), True),
    StructField('updated_at',     StringType(),  True),
])

products_df = (
    spark.read
    .option('header', 'true')
    .schema(PRODUCTS_SCHEMA)
    .csv('s3a://raw-landing/products/')
    .withColumn('ingested_at', F.lit(datetime.now(timezone.utc).isoformat()))
)

print(f'Products loaded: {products_df.count()} rows')
products_df.printSchema()
products_df.show(truncate=False)

In [ ]:
# Cell 5 — Write products as Iceberg table to Nessie catalog
(
    products_df
    .writeTo('nessie.datahive.products')
    .using('iceberg')
    .tableProperty('write.format.default', 'parquet')
    .tableProperty('write.parquet.compression-codec', 'snappy')
    .createOrReplace()
)
print('Written: nessie.datahive.products')

In [ ]:
# Cell 6 — Read orders CSV and write as partitioned Iceberg table
from pyspark.sql.types import DateType

ORDERS_SCHEMA = StructType([
    StructField('order_id',     StringType(), False),
    StructField('customer_id',  StringType(), True),
    StructField('product_id',   StringType(), True),
    StructField('quantity',     IntegerType(), True),
    StructField('unit_price',   DoubleType(),  True),
    StructField('order_status', StringType(), True),
    StructField('order_date',   StringType(), True),
])

orders_df = (
    spark.read
    .option('header', 'true')
    .schema(ORDERS_SCHEMA)
    .csv('s3a://raw-landing/orders/')
    .withColumn('order_date',   F.to_date('order_date'))
    .withColumn('ingested_at',  F.lit(datetime.now(timezone.utc).isoformat()))
)

print(f'Orders loaded: {orders_df.count()} rows')
orders_df.show(truncate=False)

(
    orders_df
    .writeTo('nessie.datahive.orders')
    .using('iceberg')
    .partitionedBy(F.col('order_status'))
    .tableProperty('write.format.default', 'parquet')
    .createOrReplace()
)
print('Written: nessie.datahive.orders (partitioned by order_status)')

In [ ]:
# Cell 7 — SQL analytics: Revenue by category
spark.sql("""
    SELECT
        p.category,
        COUNT(o.order_id)                      AS total_orders,
        ROUND(SUM(o.quantity * o.unit_price), 2) AS total_revenue,
        ROUND(AVG(o.unit_price), 2)             AS avg_price
    FROM nessie.datahive.orders  o
    JOIN nessie.datahive.products p USING (product_id)
    WHERE o.order_status = 'completed'
    GROUP BY p.category
    ORDER BY total_revenue DESC
""").show(truncate=False)

In [ ]:
# Cell 8 — Iceberg table metadata and snapshot history
print('=== All tables in nessie.datahive ===')
spark.sql('SHOW TABLES IN nessie.datahive').show()

print('=== Snapshot history for orders ===')
spark.sql('SELECT snapshot_id, committed_at, operation, summary FROM nessie.datahive.orders.snapshots').show(truncate=False)

print('=== Iceberg files for products ===')
spark.sql('SELECT file_path, file_format, record_count, file_size_in_bytes FROM nessie.datahive.products.files').show(truncate=False)

In [ ]:
# Cell 9 — Schema evolution: add columns to orders
spark.sql("""
    ALTER TABLE nessie.datahive.orders
    ADD COLUMN discount_pct DOUBLE
""")
print('Added column: discount_pct')

# Existing rows return NULL for the new column — no data rewrite
spark.sql("""
    SELECT order_id, order_status, unit_price, discount_pct
    FROM nessie.datahive.orders
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Cell 10 — Try the Glue DynamicFrame API (when running inside Glue container)
try:
    from awsglue.context import GlueContext
    from awsglue.dynamicframe import DynamicFrame

    gc = GlueContext(spark.sparkContext)

    # Read from S3/MinIO via Glue
    dyf = gc.create_dynamic_frame.from_options(
        connection_type='s3',
        connection_options={'paths': ['s3a://raw-landing/products/']},
        format='csv',
        format_options={'withHeader': True},
    )

    print('DynamicFrame schema:')
    dyf.printSchema()
    print(f'Record count: {dyf.count()}')

    # Convert to Spark DataFrame and apply transformations
    df = dyf.toDF().withColumn('price_tier',
        F.when(F.col('price') < 30, 'budget')
         .when(F.col('price') < 80, 'mid')
         .otherwise('premium')
    )
    df.show(truncate=False)

except ImportError:
    print('awsglue not available — running in plain PySpark mode')
    print('This is expected outside the Glue container')

## Next Steps

- Open the **Nessie UI** at http://localhost:19120 to browse table metadata and branches
- Open the **MinIO console** at http://localhost:9001 to see the Parquet files in `iceberg-warehouse/`
- Run the **full Glue job** from the terminal: `docker exec playground-glue python /home/glue_user/workspace/jobs/sample_glue_iceberg.py`
- Try the **EMR Spark demo** in `notebooks/emr_spark_demo.ipynb`
